# Stage 3.4: Multi-GPU Serving for Large Models

## Problem Statement
Large language models (70B+ parameters) don't fit on a single GPU. How can we efficiently distribute model computation and memory across multiple GPUs while maintaining high throughput and low latency?

## What You'll Learn
- Tensor parallelism for inference
- Model sharding strategies across GPUs
- Communication overhead analysis
- Load balancing strategies
- Ray Serve and distributed serving frameworks
- Scaling from 1 to 8 GPUs
- Cost-performance trade-offs

## Prerequisites
- Understanding of model architecture (transformers)
- Basic knowledge of distributed computing
- Familiarity with vLLM or serving frameworks

---

In [ ]:
import torch
import torch.distributed as dist
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

---

## Part 1: The Single-GPU Limitation

### Model Size vs GPU Memory

```
GPU Memory Capacity (common GPUs):
- RTX 4090:    24 GB
- A10:         24 GB
- A100 40GB:   40 GB
- A100 80GB:   80 GB
- H100:        80 GB

Model Sizes (FP16):
- Llama 2 7B:    14 GB (weights) + 5-10 GB (KV cache) = ~20 GB
- Llama 2 13B:   26 GB (weights) + 8-15 GB (KV cache) = ~40 GB
- Llama 2 70B:  140 GB (weights) + 40-60 GB (KV cache) = ~200 GB
- GPT-4 class:  ~1 TB (estimated)
```

**Problem:** Models larger than ~13B require multiple GPUs!

### Why Not Just Use Model Parallelism from Training?

Training uses different parallelism strategies:
- **Data Parallelism**: Each GPU processes different batch
- **Pipeline Parallelism**: Split model into stages
- **Tensor Parallelism**: Split individual operations

**For inference:**
- ❌ Data parallelism: Single request at a time (sequential generation)
- ❌ Pipeline parallelism: High latency (sequential stages)
- ✅ Tensor parallelism: Best for inference (parallel within layer)

In [ ]:
def calculate_model_memory(num_params_billions: float, 
                          precision: str = 'fp16',
                          batch_size: int = 1,
                          seq_length: int = 2048) -> Dict:
    """
    Calculate memory requirements for a model.
    """
    bytes_per_param = {
        'fp32': 4,
        'fp16': 2,
        'int8': 1,
        'int4': 0.5
    }
    
    # Model weights
    weights_gb = num_params_billions * bytes_per_param[precision]
    
    # KV cache (rough estimate)
    # 2 (key + value) × batch × seq_len × num_layers × hidden_dim × bytes
    num_layers = int(num_params_billions * 2)  # Rough estimate
    hidden_dim = int((num_params_billions * 1000 / num_layers) ** 0.5) * 100
    
    kv_cache_bytes = 2 * batch_size * seq_length * num_layers * hidden_dim * bytes_per_param[precision]
    kv_cache_gb = kv_cache_bytes / (1024 ** 3)
    
    # Activations (during forward pass)
    activations_gb = num_params_billions * 0.1  # Rough estimate
    
    total_gb = weights_gb + kv_cache_gb + activations_gb
    
    return {
        'weights_gb': weights_gb,
        'kv_cache_gb': kv_cache_gb,
        'activations_gb': activations_gb,
        'total_gb': total_gb
    }


# Analyze different model sizes
models = [
    ('Llama 2 7B', 7),
    ('Llama 2 13B', 13),
    ('Llama 2 70B', 70),
    ('Mixtral 8x7B', 47),  # Active parameters
    ('GPT-3 175B', 175),
]

print("\n=== MODEL MEMORY REQUIREMENTS (FP16) ===")
print(f"Batch size: 1, Sequence length: 2048\n")

gpu_capacities = {
    'A100 40GB': 40,
    'A100 80GB': 80,
    'H100': 80
}

for model_name, size in models:
    mem = calculate_model_memory(size, 'fp16')
    print(f"\n{model_name}:")
    print(f"  Weights:     {mem['weights_gb']:.1f} GB")
    print(f"  KV cache:    {mem['kv_cache_gb']:.1f} GB")
    print(f"  Activations: {mem['activations_gb']:.1f} GB")
    print(f"  Total:       {mem['total_gb']:.1f} GB")
    
    # Determine minimum GPUs needed
    print(f"  Minimum GPUs needed:")
    for gpu_name, capacity in gpu_capacities.items():
        min_gpus = int(np.ceil(mem['total_gb'] / capacity))
        if min_gpus == 1:
            print(f"    {gpu_name}: 1 GPU (fits!)")
        else:
            print(f"    {gpu_name}: {min_gpus} GPUs (requires sharding)")

---

## Part 2: Tensor Parallelism - The Key to Multi-GPU Inference

### How Tensor Parallelism Works

**Concept:** Split weight matrices across GPUs, compute in parallel

#### Column-wise Parallelism (for Linear layers)

```python
# Single GPU
output = input @ weight  # [batch, hidden] @ [hidden, hidden]

# Multi-GPU (2 GPUs)
# Split weight matrix into 2 parts
weight_1 = weight[:, :hidden//2]  # First half of columns → GPU 0
weight_2 = weight[:, hidden//2:]  # Second half of columns → GPU 1

# Compute in parallel (no communication needed!)
output_1 = input @ weight_1  # On GPU 0
output_2 = input @ weight_2  # On GPU 1

# Concatenate results
output = torch.cat([output_1, output_2], dim=-1)
```

#### Row-wise Parallelism

```python
# Multi-GPU (2 GPUs)
weight_1 = weight[:hidden//2, :]  # First half of rows → GPU 0
weight_2 = weight[hidden//2:, :]  # Second half of rows → GPU 1

output_1 = input @ weight_1  # On GPU 0
output_2 = input @ weight_2  # On GPU 1

# Need to sum results (requires all-reduce communication)
output = output_1 + output_2
```

### Transformer Layer with Tensor Parallelism

```
Standard Transformer Layer:
1. LayerNorm
2. Attention:
   - Q, K, V projections: [hidden, hidden]
   - Attention computation
   - Output projection: [hidden, hidden]
3. LayerNorm
4. MLP:
   - Up projection: [hidden, 4*hidden]
   - Activation (GELU)
   - Down projection: [4*hidden, hidden]

Tensor Parallel (2 GPUs):
GPU 0:                          GPU 1:
- Q[:, :H/2], K[:, :H/2],      - Q[:, H/2:], K[:, H/2:],
  V[:, :H/2]                     V[:, H/2:]
- Attention (half heads)       - Attention (half heads)
- Output[:H/2, :]              - Output[H/2:, :]
                               [All-Reduce sum]
- Up[:, :2H]                   - Up[:, 2H:]
- GELU                         - GELU
- Down[:2H, :]                 - Down[2H:, :]
                               [All-Reduce sum]
```

### Communication Points

**Only 2 communication points per layer:**
1. After attention output projection (all-reduce)
2. After MLP down projection (all-reduce)

**Key advantage:** Minimal communication overhead

In [ ]:
class TensorParallelLinear:
    """
    Simplified tensor parallel linear layer.
    In production, use vLLM or Megatron-LM implementations.
    """
    
    def __init__(self, 
                 in_features: int,
                 out_features: int,
                 world_size: int,
                 rank: int,
                 column_parallel: bool = True):
        """
        Args:
            in_features: Input dimension
            out_features: Output dimension
            world_size: Number of GPUs
            rank: Current GPU rank
            column_parallel: If True, split columns; else split rows
        """
        self.in_features = in_features
        self.out_features = out_features
        self.world_size = world_size
        self.rank = rank
        self.column_parallel = column_parallel
        
        if column_parallel:
            # Split output dimension across GPUs
            self.out_features_per_gpu = out_features // world_size
            self.weight_shape = (in_features, self.out_features_per_gpu)
        else:
            # Split input dimension across GPUs
            self.in_features_per_gpu = in_features // world_size
            self.weight_shape = (self.in_features_per_gpu, out_features)
        
        # Each GPU only stores its partition
        self.weight = torch.randn(self.weight_shape)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with communication when needed"""
        if self.column_parallel:
            # Column parallel: No communication needed for forward
            # Each GPU computes part of output
            output = x @ self.weight
            return output
        else:
            # Row parallel: Need all-reduce after computation
            output = x @ self.weight
            # In real implementation: dist.all_reduce(output)
            return output
    
    def memory_mb(self) -> float:
        """Memory used by this partition"""
        bytes_per_param = 2  # FP16
        total_params = self.weight.numel()
        return (total_params * bytes_per_param) / (1024 ** 2)


# Example: Llama 2 70B layer
print("\n=== TENSOR PARALLELISM EXAMPLE ===")
print("\nLlama 2 70B Single Layer (Attention QKV projection):")

hidden_dim = 8192
num_gpus_list = [1, 2, 4, 8]

print(f"\nFull weight matrix: [{hidden_dim}, {hidden_dim}]")
full_params = hidden_dim * hidden_dim
full_memory = (full_params * 2) / (1024 ** 2)  # FP16
print(f"Full memory: {full_memory:.1f} MB")

print("\n" + "="*60)
print("TENSOR PARALLEL SHARDING:")
print("="*60)

for num_gpus in num_gpus_list:
    # Simulate column-parallel layer
    layer = TensorParallelLinear(
        in_features=hidden_dim,
        out_features=hidden_dim,
        world_size=num_gpus,
        rank=0,  # Just show GPU 0
        column_parallel=True
    )
    
    memory_per_gpu = layer.memory_mb()
    
    print(f"\n{num_gpus} GPUs:")
    print(f"  Weight shape per GPU: {layer.weight_shape}")
    print(f"  Memory per GPU: {memory_per_gpu:.1f} MB")
    print(f"  Reduction from 1 GPU: {full_memory / memory_per_gpu:.1f}x")

---

## Part 3: Communication Overhead Analysis

### Communication Costs

Key operations in tensor parallelism:
1. **All-Reduce**: Sum tensors across all GPUs
2. **All-Gather**: Collect tensors from all GPUs
3. **Broadcast**: Send tensor from one GPU to all

**All-Reduce is most common** (happens 2x per transformer layer)

### Communication Time

```
Time = Latency + (Message_Size / Bandwidth)

Typical values:
- NVLink (GPU-GPU): 600 GB/s, ~5μs latency
- PCIe 4.0 x16:     32 GB/s, ~10μs latency
- InfiniBand:       200 GB/s, ~2μs latency
```

In [ ]:
def estimate_communication_time(message_size_mb: float,
                               bandwidth_gbps: float,
                               latency_us: float) -> float:
    """
    Estimate communication time for all-reduce operation.
    """
    # Convert to consistent units
    bandwidth_mbps = bandwidth_gbps * 1000
    
    # Time = latency + transfer time
    transfer_time_us = (message_size_mb / bandwidth_mbps) * 1e6
    total_time_us = latency_us + transfer_time_us
    
    return total_time_us / 1000  # Return in ms


def analyze_layer_communication(hidden_dim: int,
                               batch_size: int,
                               seq_length: int,
                               bandwidth_gbps: float,
                               latency_us: float) -> Dict:
    """
    Analyze communication overhead for one transformer layer.
    """
    # Two all-reduce operations per layer
    # 1. After attention output projection
    # 2. After MLP down projection
    
    # Message size: batch × seq_len × hidden_dim × 2 bytes (FP16)
    message_size_bytes = batch_size * seq_length * hidden_dim * 2
    message_size_mb = message_size_bytes / (1024 ** 2)
    
    # Time for one all-reduce
    single_allreduce_ms = estimate_communication_time(
        message_size_mb, bandwidth_gbps, latency_us
    )
    
    # Total communication per layer (2 all-reduces)
    total_comm_ms = 2 * single_allreduce_ms
    
    return {
        'message_size_mb': message_size_mb,
        'single_allreduce_ms': single_allreduce_ms,
        'total_comm_ms': total_comm_ms
    }


# Analyze different interconnects
print("\n=== COMMUNICATION OVERHEAD ANALYSIS ===")
print("\nLlama 2 70B (hidden_dim=8192, 80 layers)")
print("Batch size: 1, Sequence length: 1 (decode phase)\n")

hidden_dim = 8192
num_layers = 80
batch_size = 1
seq_length = 1

interconnects = [
    ('NVLink (A100)', 600, 5),
    ('NVLink (H100)', 900, 3),
    ('PCIe 4.0 x16', 32, 10),
    ('InfiniBand HDR', 200, 2),
]

for name, bandwidth, latency in interconnects:
    result = analyze_layer_communication(
        hidden_dim, batch_size, seq_length, bandwidth, latency
    )
    
    total_model_comm_ms = result['total_comm_ms'] * num_layers
    
    print(f"\n{name}:")
    print(f"  Bandwidth: {bandwidth} GB/s, Latency: {latency} μs")
    print(f"  Message size per all-reduce: {result['message_size_mb']:.3f} MB")
    print(f"  Time per all-reduce: {result['single_allreduce_ms']:.3f} ms")
    print(f"  Communication per layer: {result['total_comm_ms']:.3f} ms")
    print(f"  Total communication (80 layers): {total_model_comm_ms:.2f} ms")
    
    # Estimate compute time (very rough)
    # A100: ~300 TFLOPS FP16, 70B model ~140 TFLOP per token
    compute_time_ms = 10.0  # Rough estimate for single token
    
    overhead_pct = (total_model_comm_ms / (compute_time_ms + total_model_comm_ms)) * 100
    print(f"  Communication overhead: {overhead_pct:.1f}% of total time")

print("\n=== KEY INSIGHT ===")
print("Communication overhead is LOW with proper interconnects (NVLink)")
print("NVLink: <10% overhead")
print("PCIe: >50% overhead (avoid for multi-GPU inference!)")

---

## Part 4: Scaling Analysis - 1 to 8 GPUs

In [ ]:
def estimate_throughput(num_gpus: int,
                       model_size_b: float,
                       gpu_memory_gb: float,
                       interconnect_type: str = 'nvlink') -> Dict:
    """
    Estimate throughput for multi-GPU serving.
    """
    # Memory per GPU
    weights_per_gpu = (model_size_b * 2) / num_gpus  # FP16
    
    # Available memory for KV cache
    kv_cache_memory = gpu_memory_gb - weights_per_gpu - 5  # 5GB for activations
    kv_cache_memory = max(0, kv_cache_memory)
    
    # Estimate batch size based on available KV cache memory
    # Rough: 1MB per token per request for 70B model
    tokens_per_request = 2048
    mb_per_token = 0.5 if model_size_b < 20 else 1.0
    max_batch_size = int((kv_cache_memory * 1000) / (tokens_per_request * mb_per_token))
    max_batch_size = max(1, min(max_batch_size, 64))  # Cap at 64
    
    # Base tokens/sec for single GPU (A100 baseline)
    base_tokens_per_sec = 50 if model_size_b < 20 else 20
    
    # Communication overhead
    if interconnect_type == 'nvlink':
        comm_overhead = 1.0 + (num_gpus - 1) * 0.05  # 5% per additional GPU
    else:  # PCIe
        comm_overhead = 1.0 + (num_gpus - 1) * 0.20  # 20% per additional GPU
    
    # Throughput scales with batch size
    throughput = (base_tokens_per_sec * max_batch_size) / comm_overhead
    
    # Cost ($/hour, rough AWS pricing)
    cost_per_gpu_hour = 4.0  # Rough p4d.24xlarge pricing
    total_cost_hour = num_gpus * cost_per_gpu_hour
    
    # Cost efficiency: tokens per dollar
    tokens_per_dollar = (throughput * 3600) / total_cost_hour
    
    return {
        'num_gpus': num_gpus,
        'weights_per_gpu_gb': weights_per_gpu,
        'kv_cache_memory_gb': kv_cache_memory,
        'max_batch_size': max_batch_size,
        'throughput_tokens_sec': throughput,
        'cost_per_hour': total_cost_hour,
        'tokens_per_dollar': tokens_per_dollar,
        'comm_overhead': comm_overhead
    }


# Analyze Llama 2 70B scaling
print("\n=== MULTI-GPU SCALING ANALYSIS ===")
print("\nModel: Llama 2 70B (140GB FP16)")
print("GPU: A100 80GB with NVLink\n")

gpu_counts = [1, 2, 4, 8]
results = []

for num_gpus in gpu_counts:
    result = estimate_throughput(
        num_gpus=num_gpus,
        model_size_b=70,
        gpu_memory_gb=80,
        interconnect_type='nvlink'
    )
    results.append(result)
    
    print(f"{num_gpus} GPU(s):")
    print(f"  Weights per GPU:     {result['weights_per_gpu_gb']:.1f} GB")
    print(f"  KV cache memory:     {result['kv_cache_memory_gb']:.1f} GB")
    print(f"  Max batch size:      {result['max_batch_size']}")
    print(f"  Throughput:          {result['throughput_tokens_sec']:.1f} tokens/sec")
    print(f"  Cost:                ${result['cost_per_hour']:.2f}/hour")
    print(f"  Cost efficiency:     {result['tokens_per_dollar']:.0f} tokens/$")
    print(f"  Comm overhead:       {(result['comm_overhead'] - 1) * 100:.1f}%")
    print()

# Visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# Throughput
throughputs = [r['throughput_tokens_sec'] for r in results]
ax1.bar(gpu_counts, throughputs, color='#3498db', alpha=0.8, edgecolor='black')
ax1.set_xlabel('Number of GPUs', fontsize=12)
ax1.set_ylabel('Throughput (tokens/sec)', fontsize=12)
ax1.set_title('Throughput vs Number of GPUs', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

for i, (gpus, throughput) in enumerate(zip(gpu_counts, throughputs)):
    ax1.text(gpus, throughput, f'{throughput:.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Cost
costs = [r['cost_per_hour'] for r in results]
ax2.bar(gpu_counts, costs, color='#e74c3c', alpha=0.8, edgecolor='black')
ax2.set_xlabel('Number of GPUs', fontsize=12)
ax2.set_ylabel('Cost ($/hour)', fontsize=12)
ax2.set_title('Cost vs Number of GPUs', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

for i, (gpus, cost) in enumerate(zip(gpu_counts, costs)):
    ax2.text(gpus, cost, f'${cost:.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Cost efficiency
efficiencies = [r['tokens_per_dollar'] for r in results]
ax3.bar(gpu_counts, efficiencies, color='#2ecc71', alpha=0.8, edgecolor='black')
ax3.set_xlabel('Number of GPUs', fontsize=12)
ax3.set_ylabel('Tokens per Dollar', fontsize=12)
ax3.set_title('Cost Efficiency vs Number of GPUs', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

for i, (gpus, eff) in enumerate(zip(gpu_counts, efficiencies)):
    ax3.text(gpus, eff, f'{eff:.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Speedup vs ideal
ideal_speedup = gpu_counts
actual_speedup = [throughputs[i] / throughputs[0] for i in range(len(throughputs))]
efficiency = [actual / ideal * 100 for actual, ideal in zip(actual_speedup, ideal_speedup)]

ax4.plot(gpu_counts, ideal_speedup, marker='o', linewidth=2, markersize=8,
        label='Ideal (Linear)', color='gray', linestyle='--')
ax4.plot(gpu_counts, actual_speedup, marker='s', linewidth=2, markersize=8,
        label='Actual', color='#9b59b6')
ax4.set_xlabel('Number of GPUs', fontsize=12)
ax4.set_ylabel('Speedup', fontsize=12)
ax4.set_title('Scaling Efficiency', fontsize=14, fontweight='bold')
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3)

for i, (gpus, speedup, eff) in enumerate(zip(gpu_counts, actual_speedup, efficiency)):
    ax4.text(gpus, speedup, f'{eff:.0f}%',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('multi_gpu_scaling.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== SCALING INSIGHTS ===")
print(f"Linear speedup (ideal): {ideal_speedup[-1]}x")
print(f"Actual speedup (8 GPUs): {actual_speedup[-1]:.2f}x")
print(f"Scaling efficiency: {efficiency[-1]:.1f}%")
print(f"\nBest cost efficiency: {max(efficiencies):.0f} tokens/$ at {gpu_counts[efficiencies.index(max(efficiencies))]} GPU(s)")

---

## Part 5: Production Serving with vLLM

### Single-Node Multi-GPU Setup

```python
from vllm import LLM, SamplingParams

# Initialize vLLM with tensor parallelism
llm = LLM(
    model="meta-llama/Llama-2-70b-hf",
    tensor_parallel_size=4,  # Use 4 GPUs
    dtype="float16",
    gpu_memory_utilization=0.9,
    max_num_seqs=256,  # Continuous batching
    enable_prefix_caching=True
)

# Generate
prompts = ["Tell me about AI", "What is quantum computing?"]
sampling_params = SamplingParams(temperature=0.7, max_tokens=100)

outputs = llm.generate(prompts, sampling_params)
```

### Multi-Node Setup (Ray Serve)

```python
import ray
from ray import serve
from vllm import LLM, SamplingParams

# Initialize Ray
ray.init(address="auto")  # Connect to Ray cluster

@serve.deployment(
    ray_actor_options={
        "num_gpus": 4,  # 4 GPUs per replica
    },
    num_replicas=2,  # 2 replicas = 8 GPUs total
)
class LLMService:
    def __init__(self):
        self.llm = LLM(
            model="meta-llama/Llama-2-70b-hf",
            tensor_parallel_size=4,  # 4 GPUs per replica
        )
    
    async def __call__(self, request):
        prompts = [request.query_params["prompt"]]
        sampling_params = SamplingParams(temperature=0.7)
        
        outputs = self.llm.generate(prompts, sampling_params)
        return outputs[0].outputs[0].text

# Deploy
serve.run(LLMService.bind())
```

### Load Balancing Strategies

1. **Round Robin**: Simple, fair distribution
2. **Least Loaded**: Send to replica with fewest active requests
3. **Shortest Queue**: Consider request queue length
4. **Weighted**: Based on GPU capacity/performance

```python
# Example with Ray Serve
@serve.deployment(
    autoscaling_config={
        "min_replicas": 1,
        "max_replicas": 4,
        "target_num_ongoing_requests_per_replica": 10,
    },
    max_concurrent_queries=100,
)
class AutoScalingLLM:
    # Automatically scales based on load
    pass
```

---

## Part 6: Cost-Performance Trade-offs

In [ ]:
def compare_deployment_options(model_size_b: float, 
                              target_throughput: float) -> List[Dict]:
    """
    Compare different deployment configurations to achieve target throughput.
    """
    options = []
    
    # Option 1: Minimum GPUs (tensor parallel)
    min_gpus_tp = max(2, int(np.ceil(model_size_b * 2 / 80)))  # 80GB A100
    
    result_tp = estimate_throughput(min_gpus_tp, model_size_b, 80, 'nvlink')
    num_replicas_tp = int(np.ceil(target_throughput / result_tp['throughput_tokens_sec']))
    total_gpus_tp = min_gpus_tp * num_replicas_tp
    total_cost_tp = result_tp['cost_per_hour'] * num_replicas_tp
    
    options.append({
        'name': f'Minimal TP ({min_gpus_tp} GPUs/replica)',
        'gpus_per_replica': min_gpus_tp,
        'num_replicas': num_replicas_tp,
        'total_gpus': total_gpus_tp,
        'throughput': result_tp['throughput_tokens_sec'] * num_replicas_tp,
        'cost_per_hour': total_cost_tp,
        'latency_relative': 1.0
    })
    
    # Option 2: More GPUs per replica (better latency)
    if model_size_b >= 50:
        more_gpus_tp = min_gpus_tp * 2
        result_more = estimate_throughput(more_gpus_tp, model_size_b, 80, 'nvlink')
        num_replicas_more = int(np.ceil(target_throughput / result_more['throughput_tokens_sec']))
        total_gpus_more = more_gpus_tp * num_replicas_more
        total_cost_more = result_more['cost_per_hour'] * num_replicas_more
        
        options.append({
            'name': f'More TP ({more_gpus_tp} GPUs/replica)',
            'gpus_per_replica': more_gpus_tp,
            'num_replicas': num_replicas_more,
            'total_gpus': total_gpus_more,
            'throughput': result_more['throughput_tokens_sec'] * num_replicas_more,
            'cost_per_hour': total_cost_more,
            'latency_relative': 0.8  # Better latency with more parallelism
        })
    
    return options


# Compare deployment options
print("\n=== DEPLOYMENT OPTIONS COMPARISON ===")
print("\nTarget: 1000 tokens/sec throughput")
print("Model: Llama 2 70B\n")

options = compare_deployment_options(
    model_size_b=70,
    target_throughput=1000
)

for i, option in enumerate(options, 1):
    print(f"Option {i}: {option['name']}")
    print(f"  Configuration: {option['num_replicas']} replicas × {option['gpus_per_replica']} GPUs")
    print(f"  Total GPUs: {option['total_gpus']}")
    print(f"  Throughput: {option['throughput']:.0f} tokens/sec")
    print(f"  Cost: ${option['cost_per_hour']:.2f}/hour")
    print(f"  Relative latency: {option['latency_relative']:.2f}x")
    print()

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = [opt['name'] for opt in options]
costs = [opt['cost_per_hour'] for opt in options]
throughputs = [opt['throughput'] for opt in options]
latencies = [opt['latency_relative'] for opt in options]

# Cost comparison
colors = ['#3498db', '#e74c3c']
bars = ax1.bar(range(len(names)), costs, color=colors, alpha=0.8, edgecolor='black')
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels(names, rotation=15, ha='right')
ax1.set_ylabel('Cost ($/hour)', fontsize=12)
ax1.set_title('Deployment Cost Comparison', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

for bar, cost in zip(bars, costs):
    ax1.text(bar.get_x() + bar.get_width()/2, cost,
            f'${cost:.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Latency vs cost scatter
ax2.scatter(costs, latencies, s=200, c=colors, alpha=0.8, edgecolors='black', linewidth=2)

for i, (cost, latency, name) in enumerate(zip(costs, latencies, names)):
    ax2.annotate(name, (cost, latency), 
                xytext=(10, 10), textcoords='offset points',
                fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax2.set_xlabel('Cost ($/hour)', fontsize=12)
ax2.set_ylabel('Relative Latency (lower is better)', fontsize=12)
ax2.set_title('Cost vs Latency Trade-off', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.invert_yaxis()  # Lower latency is better

plt.tight_layout()
plt.savefig('deployment_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== RECOMMENDATION ===")
best_cost = min(options, key=lambda x: x['cost_per_hour'])
best_latency = min(options, key=lambda x: x['latency_relative'])

print(f"Best cost: {best_cost['name']} - ${best_cost['cost_per_hour']:.2f}/hour")
print(f"Best latency: {best_latency['name']} - {best_latency['latency_relative']:.2f}x relative latency")

---

## Part 7: Key Takeaways

### When to Use Multi-GPU Inference

1. **Model doesn't fit on single GPU**
   - 70B+ models require multiple GPUs
   - Even with quantization (INT4), 70B needs 2+ GPUs

2. **Need higher throughput**
   - More GPUs = larger batch sizes
   - Better for high-traffic applications

3. **Lower latency requirements**
   - More tensor parallelism = faster per-token generation
   - Trade cost for latency

### Tensor Parallelism Best Practices

1. **Use NVLink whenever possible**
   - 10x faster than PCIe
   - <10% communication overhead

2. **Power of 2 GPU counts**
   - 2, 4, 8 GPUs for clean matrix splits
   - Easier load balancing

3. **Balance with replication**
   - Don't use maximum TP if you need throughput
   - Example: 8 GPUs = 4 replicas × 2 GPUs > 1 replica × 8 GPUs

### Cost Optimization

| Strategy | When to Use | Trade-off |
|----------|-------------|------------|
| Minimal TP (2 GPUs) | Cost-sensitive, moderate latency OK | Lowest cost, higher latency |
| Balanced (4 GPUs) | Production default | Balanced cost/performance |
| Max TP (8 GPUs) | Latency-critical applications | Highest cost, lowest latency |
| Replication | High throughput needed | Linear cost scaling |

### Scaling Efficiency

```
Observed scaling efficiency (with NVLink):
- 2 GPUs: ~95% efficiency
- 4 GPUs: ~90% efficiency
- 8 GPUs: ~85% efficiency

Communication overhead increases with GPU count,
but remains manageable with proper interconnect.
```

### Production Recommendations

1. **Start with minimum viable TP**
   - Just enough GPUs to fit model
   - Scale out with replication for throughput

2. **Monitor metrics**
   - GPU utilization per device
   - Communication time vs compute time
   - Request latency distribution

3. **Use established frameworks**
   - vLLM: Best for general inference
   - TensorRT-LLM: Maximum performance
   - Ray Serve: Multi-node orchestration

---

## References & Next Steps

### Related Notebooks
- **Notebook 30**: Continuous batching
- **Notebook 31**: PagedAttention (memory management)
- **Notebook 33**: Prefix caching

### Framework Documentation
- **vLLM**: https://docs.vllm.ai/en/latest/serving/distributed_serving.html
- **Ray Serve**: https://docs.ray.io/en/latest/serve/
- **Megatron-LM**: https://github.com/NVIDIA/Megatron-LM
- **TensorRT-LLM**: https://github.com/NVIDIA/TensorRT-LLM

### Papers
- "Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism"
- "Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM"

### LLM Inference Roadmap
- See `LLM_INFERENCE_ROADMAP.md` - Stage 3: Advanced Serving

---

## Summary

Multi-GPU serving enables **large model inference** through:

1. **Tensor Parallelism**: Split operations across GPUs
2. **Minimal Communication**: Only 2 all-reduce ops per layer
3. **Near-Linear Scaling**: 85-95% efficiency with NVLink
4. **Flexible Deployment**: Balance TP and replication for cost/performance

**Key principle:** Use minimum GPUs for memory, scale with replication for throughput.

Combined with continuous batching, PagedAttention, and prefix caching, multi-GPU serving enables production-scale deployment of large models.

**Next:** Stage 4 - Speed Optimization (speculative decoding, quantization, kernel optimization)